<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-00-prerequisites/lesson-0.4-profiling-python-ai/practice/practice-lab-0_4.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Practice Lab: Profiling Python AI Workloads

**Netsetos GenAI Course** | Module 0 · Lesson 0.4 | ⏱ ~60-90 min

**Prerequisites:** Python 3.10+ (Google Colab recommended). Modules: cProfile, pstats, time, numpy.

### 🎯 You will:
1. Profile a simulated pipeline with cProfile and find the dominant stage
2. Find line-level hotspots and apply the pre-normalize optimization
3. Build an ASCII flame graph from cProfile data
4. Classify time into Python / C-extension / I/O categories
5. Build a RAG stage profiler with percentile stats
6. Combine all tools into a production profiling dashboard

---

## Exercise 1 (Easy): cProfile a Simulated Pipeline

**📖 Concept:** cProfile records ncalls, tottime (THIS function only), cumtime (total with subcalls). Sort by `cumulative` to find slow pipelines, `tottime` to find actual bottlenecks.

**📝 Task:**
1. Build a 4-stage pipeline with `time.sleep()`: parse(100ms), embed(200ms), search(50ms), llm(500ms)
2. Profile 10 iterations with cProfile
3. Sort by cumulative — print top 10
4. Calculate % of total for each stage

**📤 Expected:** call_llm ~59%, embed_query ~24%, parse_docs ~12%, search_index ~6%

In [ ]:
# ✏️ YOUR CODE HERE
import cProfile, pstats, time

def parse_docs(): time.sleep(0.1); return ["doc1"]
def embed_query(q): time.sleep(0.2); return [0.1] * 1536
def search_index(emb): time.sleep(0.05); return ["r1"]
def call_llm(ctx, q): time.sleep(0.5); return f"Answer: {q}"

def rag_pipeline(q):
    return call_llm(search_index(embed_query(q)), q)

# TODO: profile 10 iterations, sort, calculate %

<details><summary>💡 Hint</summary>

```python
profiler = cProfile.Profile()
profiler.enable()
for _ in range(10): rag_pipeline("q")
profiler.disable()
pstats.Stats(profiler).sort_stats('cumulative').print_stats(10)
```
</details>

**✅ Criteria:** cProfile wraps pipeline, top functions sorted, call_llm identified as dominant, % breakdown calculated

<details><summary>✅ Solution</summary>



In [ ]:
import cProfile, pstats, time

def parse_docs(): time.sleep(0.1); return ["doc1"]
def embed_query(q): time.sleep(0.2); return [0.1]*1536
def search_index(emb): time.sleep(0.05); return ["r1"]
def call_llm(ctx, q): time.sleep(0.5); return f"Answer: {q}"
def rag_pipeline(q): return call_llm(search_index(embed_query(q)), q)

profiler = cProfile.Profile()
profiler.enable()
for _ in range(10): rag_pipeline("What is RAG?")
profiler.disable()

print('=== Sorted by CUMULATIVE ===')
pstats.Stats(profiler).sort_stats('cumulative').print_stats(10)

stages = {'call_llm': 5.0, 'embed_query': 2.0, 'parse_docs': 1.0, 'search_index': 0.5}
total = sum(stages.values())
print('=== Breakdown ===')
for s, t in sorted(stages.items(), key=lambda x: -x[1]):
    print(f'  {s:<15} {t:.1f}s ({t/total*100:.0f}%)')

</details>

---

## Exercise 2 (Easy): Line-Level Hotspot Finder

**📖 Concept:** Once cProfile finds the hot function, add `time.perf_counter()` between lines to find which LINE is slow. Fix: pre-normalize docs at index time.

**📝 Task:**
1. Write `cosine_search()` with line-level timing
2. Run with 10K docs × 768-dim
3. Find hotspot (doc normalization)
4. Pre-normalize, show speedup

**📤 Expected:** doc_norms ~40% before fix → 0% after fix, ~2x speedup

In [ ]:
# ✏️ YOUR CODE HERE
import numpy as np, time
np.random.seed(42)
docs = np.random.randn(10000, 768).astype(np.float32)
query = np.random.randn(768).astype(np.float32)

def cosine_search(query, docs):
    # TODO: add timing between lines
    query_norm = query / np.linalg.norm(query)
    doc_norms = np.linalg.norm(docs, axis=1, keepdims=True)
    docs_normed = docs / doc_norms
    scores = docs_normed @ query_norm
    return np.argsort(scores)[-10:][::-1]

# TODO: run, find hotspot, apply fix, show speedup

<details><summary>💡 Hint</summary>

```python
t0 = time.perf_counter()
query_norm = query / np.linalg.norm(query)
t1 = time.perf_counter()
# ... each line, print (t_end-t_start)/total*100
```
</details>

**✅ Criteria:** Per-line %, hotspot found, pre-normalize applied, speedup measured

<details><summary>✅ Solution</summary>



In [ ]:
import numpy as np, time
np.random.seed(42)
docs = np.random.randn(10000, 768).astype(np.float32)
query = np.random.randn(768).astype(np.float32)

print('=== BEFORE FIX ===')
t0 = time.perf_counter()
qn = query / np.linalg.norm(query)
t1 = time.perf_counter()
dn = np.linalg.norm(docs, axis=1, keepdims=True)
t2 = time.perf_counter()
dn2 = docs / dn
t3 = time.perf_counter()
sc = dn2 @ qn
t4 = time.perf_counter()
top = np.argsort(sc)[-10:][::-1]
t5 = time.perf_counter()
tot = t5-t0
for label, a, b in [('query_norm',t0,t1),('doc_norms',t1,t2),('divide',t2,t3),('matmul',t3,t4),('argsort',t4,t5)]:
    print(f'  {label:<12} {(b-a)/tot*100:5.1f}%{"  ← hotspot" if label=="doc_norms" else ""}')
print(f'  Total: {tot*1000:.1f}ms')

print('\n=== AFTER FIX ===')
docs_pre = docs / np.linalg.norm(docs, axis=1, keepdims=True)
t0 = time.perf_counter()
qn = query / np.linalg.norm(query)
sc = docs_pre @ qn
top = np.argsort(sc)[-10:][::-1]
tot2 = time.perf_counter()-t0
print(f'  Total: {tot2*1000:.1f}ms')
print(f'  Speedup: {tot/tot2:.1f}x')

</details>

---

## Exercise 3 (Medium): Flame Graph Simulator

**📖 Concept:** Flame graphs: x-axis = time proportion (wider = slower), y-axis = call depth. py-spy generates them from running processes. Here we simulate with ASCII art.

**📝 Task:**
1. Build nested call tree (5+ functions, 3 levels)
2. Profile with cProfile, extract cumtimes
3. Generate ASCII flame graph with proportional bars
4. Identify hottest leaf function

In [ ]:
# ✏️ YOUR CODE HERE
import cProfile, pstats, io, time

def helper_a1(): time.sleep(0.3)
def helper_a2(): time.sleep(0.1)
def helper_b1(): time.sleep(0.5)
def stage_a(): helper_a1(); helper_a2()
def stage_b(): helper_b1()
def main(): stage_a(); stage_b()

# TODO: profile, extract cumtimes, ASCII flame graph

<details><summary>💡 Hint</summary>

```python
for key, value in stats.stats.items():
    func_name = key[2]; cumtime = value[3]
# bar = '█' * int(cumtime/max_time * 40)
```
</details>

**✅ Criteria:** Nested tree profiled, cumtimes extracted, ASCII bars proportional, hottest leaf identified

<details><summary>✅ Solution</summary>



In [ ]:
import cProfile, pstats, time

def helper_a1(): time.sleep(0.3)
def helper_a2(): time.sleep(0.1)
def helper_b1(): time.sleep(0.5)
def stage_a(): helper_a1(); helper_a2()
def stage_b(): helper_b1()
def main(): stage_a(); stage_b()

profiler = cProfile.Profile()
profiler.enable()
for _ in range(5): main()
profiler.disable()

stats = pstats.Stats(profiler)
targets = {'main','stage_a','stage_b','helper_a1','helper_a2','helper_b1'}
ft = {}
for key, val in stats.stats.items():
    if key[2] in targets:
        ft[key[2]] = val[3]

mx = max(ft.values()) if ft else 1
print('ASCII Flame Graph:')
print('-'*55)
for name, t in sorted(ft.items(), key=lambda x: -x[1]):
    bar = '█' * int(t/mx*40)
    tag = ' ← hottest leaf' if name == 'helper_b1' else ''
    print(f'  {name:<12} {bar:<40} {t/mx*100:5.1f}% ({t:.2f}s){tag}')
print('-'*55)

</details>

---

## Exercise 4 (Medium): Scalene-Style CPU Split Analyzer

**📖 Concept:** Scalene separates Python (bytecode), C (NumPy native), I/O (waits). Knowing which dominates tells you WHAT to optimize.

**📝 Task:**
1. Write 3 functions: pure Python loop, NumPy SVD, time.sleep
2. Time each, classify Python% / C% / I/O%
3. Print Scalene-style report
4. Fix Python bottleneck with NumPy, show speedup

In [ ]:
# ✏️ YOUR CODE HERE
import time, numpy as np

def pure_python_work(n=1_000_000):
    total = 0
    for i in range(n): total += i*i
    return total

def numpy_work(size=1000):
    return np.linalg.svd(np.random.randn(size,size).astype(np.float32), compute_uv=False)

def io_work(s=0.5): time.sleep(s); return 'done'

# TODO: time each, classify, report, fix Python bottleneck

<details><summary>💡 Hint</summary>

Fix: `arr = np.arange(n, dtype=np.int64); return int(np.sum(arr * arr))`

</details>

**✅ Criteria:** 3 categories timed, Scalene-style table, Python→NumPy fix, speedup shown

<details><summary>✅ Solution</summary>



In [ ]:
import time, numpy as np

def pure_python_work(n=1_000_000):
    total = 0
    for i in range(n): total += i*i
    return total

def numpy_work(size=1000):
    return np.linalg.svd(np.random.randn(size,size).astype(np.float32), compute_uv=False)

def io_work(s=0.5): time.sleep(s); return 'done'

timings = {}
for name, func in [('Python', pure_python_work), ('C/native', numpy_work), ('I/O', io_work)]:
    s = time.perf_counter()
    func()
    timings[name] = (time.perf_counter()-s)*1000

total = sum(timings.values())
print('Scalene-Style Report:')
for cat, t in sorted(timings.items(), key=lambda x: -x[1]):
    action = {'Python':'← vectorize!','I/O':'← async/batch','C/native':'(optimized)'}.get(cat,'')
    print(f'  {cat:<10} {t:>8.1f}ms  {t/total*100:>5.1f}%  {action}')

print('\nFix: Python → NumPy')
s = time.perf_counter()
arr = np.arange(1_000_000, dtype=np.int64)
int(np.sum(arr*arr))
fixed = (time.perf_counter()-s)*1000
print(f'  Before: {timings["Python"]:.1f}ms → After: {fixed:.2f}ms → {timings["Python"]/fixed:.0f}x speedup')

</details>

---

## Exercise 5 (Hard): RAG Pipeline Stage Profiler

**📖 Concept:** Standard profilers don't track GenAI metrics. `RAGProfiler` uses a context manager to time stages, then computes p50/p95/p99.

**📝 Task:**
1. Build `RAGProfiler` with `time_stage()` context manager, `percentile()`, `report()`
2. Run 100 iterations of 4-stage pipeline with random variance
3. Print p50/p95/p99 per stage
4. Identify highest-variance stage

In [ ]:
# ✏️ YOUR CODE HERE
from collections import defaultdict
import time, random

class RAGProfiler:
    def __init__(self): self.stages = defaultdict(list)
    def time_stage(self, name): pass  # TODO
    def percentile(self, data, pct): pass  # TODO
    def report(self): pass  # TODO

profiler = RAGProfiler()
for _ in range(100):
    with profiler.time_stage('parse'): time.sleep(random.uniform(0.005,0.015))
    with profiler.time_stage('embed'): time.sleep(random.uniform(0.01,0.03))
    with profiler.time_stage('search'): time.sleep(random.uniform(0.003,0.008))
    with profiler.time_stage('llm_call'): time.sleep(random.uniform(0.04,0.12))
profiler.report()

<details><summary>💡 Hint</summary>

```python
def time_stage(self, name):
    parent = self
    class Timer:
        def __enter__(self): self.start = time.perf_counter()
        def __exit__(self, *a): parent.stages[name].append((time.perf_counter()-self.start)*1000)
    return Timer()
```
</details>

**✅ Criteria:** Context manager works, percentiles correct, per-stage + total stats, highest variance identified

<details><summary>✅ Solution</summary>



In [ ]:
from collections import defaultdict
import time, random

class RAGProfiler:
    def __init__(self): self.stages = defaultdict(list)

    def time_stage(self, name):
        parent = self
        class Timer:
            def __enter__(self): self.start = time.perf_counter(); return self
            def __exit__(self, *a): parent.stages[name].append((time.perf_counter()-self.start)*1000)
        return Timer()

    def percentile(self, data, pct):
        s = sorted(data)
        return s[min(int(len(s)*pct/100), len(s)-1)]

    def report(self):
        print(f'{"Stage":<12} {"p50":>8} {"p95":>8} {"p99":>8} {"Var":>8}')
        print('-'*42)
        max_var, max_s = 0, ''
        for stage in self.stages:
            p50 = self.percentile(self.stages[stage], 50)
            p95 = self.percentile(self.stages[stage], 95)
            p99 = self.percentile(self.stages[stage], 99)
            var = p95/p50 if p50 > 0 else 0
            if var > max_var: max_var, max_s = var, stage
            print(f'  {stage:<10} {p50:>7.1f} {p95:>7.1f} {p99:>7.1f} {var:>7.2f}x')
        print(f'\n  Highest variance: {max_s} ({max_var:.2f}x)')

random.seed(42)
profiler = RAGProfiler()
for _ in range(100):
    with profiler.time_stage('parse'): time.sleep(random.uniform(0.005,0.015))
    with profiler.time_stage('embed'): time.sleep(random.uniform(0.01,0.03))
    with profiler.time_stage('search'): time.sleep(random.uniform(0.003,0.008))
    with profiler.time_stage('llm_call'): time.sleep(random.uniform(0.04,0.12))

print('RAG Profile (100 runs):')
profiler.report()

</details>

---

## Exercise 6 (Hard): Production Profiling Dashboard — All Tools

**📖 Concept:** Combine cProfile overview + line timing + category split + RAGProfiler + token throughput into one diagnostic report.

**📝 Task:**
1. cProfile → top 5 by cumtime
2. Line timing on hottest function
3. Python / C / I/O split
4. RAGProfiler for 50 runs → p50/p95
5. Token throughput: tokens/sec, cost/query
6. 3 optimization recommendations

In [ ]:
# ✏️ YOUR CODE HERE
import cProfile, pstats, time, random
import numpy as np
from collections import defaultdict

def parse_documents(n=5):
    return [' '.join([f'w{j}' for j in range(200)]) for _ in range(n)]

def embed_documents(docs):
    return np.random.randn(len(docs), 768).astype(np.float32)

def vector_search(q, d, k=3):
    s = d @ q / (np.linalg.norm(d,axis=1)*np.linalg.norm(q))
    return np.argsort(s)[-k:][::-1]

def call_llm(ctx, q):
    time.sleep(random.uniform(0.05,0.15))
    return f'Answer to {q}', random.randint(50,200)

# TODO: full profiling dashboard

<details><summary>💡 Hint</summary>

Run each analysis sequentially: cProfile first, then category split, then RAGProfiler, then tokens/sec = total_tokens / total_time.

</details>

**✅ Criteria:** cProfile overview, line timing, category split, RAGProfiler p50/p95, token throughput, 3 recommendations

<details><summary>✅ Solution</summary>



In [ ]:
import cProfile, pstats, time, random
import numpy as np
from collections import defaultdict

def parse_documents(n=5):
    return [' '.join([f'w{j}' for j in range(200)]) for _ in range(n)]
def embed_documents(docs):
    return np.random.randn(len(docs), 768).astype(np.float32)
def vector_search(q, d, k=3):
    s = d @ q / (np.linalg.norm(d,axis=1)*np.linalg.norm(q))
    return np.argsort(s)[-k:][::-1]
def call_llm(ctx, q):
    time.sleep(random.uniform(0.05,0.15))
    return f'Answer: {q}', random.randint(50,200)

def pipeline(q):
    docs = parse_documents()
    embs = embed_documents(docs)
    qe = np.random.randn(768).astype(np.float32)
    vector_search(qe, embs)
    return call_llm(docs, q)

random.seed(42); np.random.seed(42)

print('═'*50)
print('  PROFILING DIAGNOSTIC REPORT')
print('═'*50)

# 1. cProfile
print('\n--- 1. cProfile (10 runs) ---')
pr = cProfile.Profile()
pr.enable()
for _ in range(10): pipeline('q')
pr.disable()
pstats.Stats(pr).sort_stats('cumulative').print_stats(8)

# 2. Category split
print('--- 2. Category Split ---')
cat = {}
s=time.perf_counter(); parse_documents(); cat['Python']=( time.perf_counter()-s)*1000
s=time.perf_counter(); embed_documents(['t']*5); cat['C/native']=(time.perf_counter()-s)*1000
s=time.perf_counter(); call_llm([],'t'); cat['I/O']=(time.perf_counter()-s)*1000
tot=sum(cat.values())
for c,t in sorted(cat.items(), key=lambda x:-x[1]):
    print(f'  {c:<10} {t:>7.1f}ms ({t/tot*100:.0f}%)')

# 3. RAGProfiler
print('\n--- 3. Stage Profiler (50 runs) ---')
class RP:
    def __init__(self): self.stages=defaultdict(list)
    def ts(self, n):
        p=self
        class T:
            def __enter__(s): s.t=time.perf_counter()
            def __exit__(s,*a): p.stages[n].append((time.perf_counter()-s.t)*1000)
        return T()
    def pct(self, d, p): s=sorted(d); return s[min(int(len(s)*p/100),len(s)-1)]

rp=RP(); toks=[]
for _ in range(50):
    with rp.ts('parse'): parse_documents()
    with rp.ts('embed'): embed_documents(['t']*5); qe=np.random.randn(768).astype(np.float32)
    with rp.ts('search'): vector_search(qe, np.random.randn(5,768).astype(np.float32))
    with rp.ts('llm'): _,tk=call_llm([],'q'); toks.append(tk)

for st in rp.stages:
    print(f'  {st:<8} p50={rp.pct(rp.stages[st],50):>7.1f}ms  p95={rp.pct(rp.stages[st],95):>7.1f}ms')

# 4. Tokens
print('\n--- 4. Token Throughput ---')
total_t = sum(sum(rp.stages[s]) for s in rp.stages)/1000
print(f'  Avg tokens: {np.mean(toks):.0f}/query')
print(f'  Throughput: {sum(toks)/total_t:.0f} tok/s')
print(f'  Cost/query: ${np.mean(toks)/1000*0.01:.5f}')

print('\n═'*50)
print('  RECOMMENDATIONS')
print('═'*50)
print('  1. Cache/batch LLM calls (dominates latency)')
print('  2. Pre-normalize embeddings (saves ~40% of search)')
print('  3. Use async for LLM calls (Lesson 0.1 pattern)')

</details>

---

## 🎉 Done!

You've mastered all 5 profiling tools:
- **cProfile** — function-level overview
- **line_profiler** — line-by-line hotspot detection
- **py-spy** — zero-code-change production profiling
- **Scalene** — Python / C / GPU time separation
- **GenAI Profiling** — TTFT, tokens/sec, stage p50/p95

The golden rule: **measure before optimizing.** Next: **Module 1.**